In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt
import pandas as pd
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()
import numpy as np
import seaborn as sn
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima_model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df.isnull().sum()


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Filter records for store 1 and item 1 -> to be able to scale to other items in the future
df = df[df['store'] == 1]
df = df[df['item'] == 1]

df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d') # convert date column to datatime object

# Create Date-related Features to be used for EDA and Supervised ML: Regression
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.weekday
df['weekday'] = np.where(df.weekday == 0, 7, df.weekday)

# Split the series to predict the last 3 months of 2017
temp_df = df.set_index('date')
train_df = temp_df.loc[:'2017-09-30'].reset_index(drop=False)
test_df = temp_df.loc['2017-10-01':].reset_index(drop=False)

train_df.head()

In [ ]:
test_df.head()


In [ ]:
plot = sn.boxplot(x='weekday', y='sales', data=df)
_ = plot.set(title='Weekly sales distribution')

# Sales Over time

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Convert 'date' column to datetime
df['date'] = pd.to_datetime(df['date'])

# Aggregate sales by date
sales_over_time = df.groupby('date')['sales'].sum().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=sales_over_time, x='date', y='sales')
plt.title('Total Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.show()



# Distribution of Sales

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['sales'], bins=30, kde=True)
plt.title('Distribution of Sales')
plt.xlabel('Sales')
plt.ylabel('Frequency')
plt.show()


In [ ]:
monthly_agg = df.groupby('month')['sales'].sum().reset_index()
fig, axs = plt.subplots(nrows=2, figsize=(9,7))
sn.boxplot(x='month', y='sales', data=df, ax=axs[0])
_ = sn.lineplot(x='month', y='sales', data=monthly_agg, ax=axs[1])

Inference: The number of sales gradually ascends in the first half of the year starting February (2), peaks in July (7), and then gradually descends, before slightly increasing in November (11) and then dropping again in December (12).



In [ ]:

yearly_agg = df.groupby('year')['sales'].sum().reset_index()
fig, axs = plt.subplots(nrows=2, figsize=(9,7))
sn.boxplot(x='year', y='sales', data=df, ax=axs[0])
_ = sn.lineplot(x='year', y='sales', data=yearly_agg, ax=axs[1])

 Autoregressive Integrated Moving Average - ARIMA Model


Step 1: Check stationarity
Before going any further into our analysis, our series has to be made stationary.

Stationarity is the property of exhibiting constant statistical properties (mean, variance, autocorrelation, etc.). If the mean of a time-series increases over time, then it’s not stationary.

The mean across many time periods is only informative if the expected value is the same across those time periods. If these population parameters can vary, what are we really estimating by taking an average across time?

Stationarity requires that the statistical properties must be the same across time, making the sample average a reasonable way to estimate them.

Methods to Check Stationarity

Plotting rolling statistics: Plotting rolling means and variances is a first good way to visually inspect our series. If the rolling statistics exhibit a clear trend (upwards or downwards) and show varying variance (increasing or decreasing amplitude), then you might conclude that the series is very likely not to be stationary.

Augmented Dickey-Fuller Test: This test is used to assess whether or not a time-series is stationary. It gives a result called a “test-statistic”, based on which you can say, with different levels (or percentage) of confidence, if the time-series is stationary or not. The test statistic is expected to be negative; therefore, it has to be more negative(less) than the critical value for the hypothesis to be rejected and conclude that series is stationary.

ACF and PACF plots: An autocorrelation (ACF) plot represents the autocorrelation of the series with lags of itself. A partial autocorrelation (PACF) plot represents the amount of correlation between a series and a lag of itself that is not explained by correlations at all lower-order lags. Ideally, we want no correlation between the series and lags of itself. Graphically speaking, we would like all the spikes to fall in the blue region.

In [ ]:
arima_df = train_df[['date', 'sales']].set_index('date')
arima_test_df = test_df[['date', 'sales']].set_index('date')

def test_stationarity(timeseries):
    # Plotting rolling statistics
    rollmean = timeseries.rolling(window=365).mean()
    rollstd = timeseries.rolling(window=365).std()

    plt.figure(figsize=(14,7))
    plt.plot(timeseries, color='skyblue', label='Original Series')
    plt.plot(rollmean, color='black', label='Rolling Mean')
    plt.plot(rollstd, color='red', label='Rolling Std')
    plt.legend(loc='best')
    plt.xlabel('date')
    plt.ylabel('sales')
    plt.show()

    # Augmented Dickey-Fuller Test
    adfuller_test = adfuller(timeseries, autolag='AIC')
    print("Test statistic = {:.3f}".format(adfuller_test[0]))
    print("P-value = {:.3f}".format(adfuller_test[1]))
    print("Critical values :")

    for key, value in adfuller_test[4].items():
        print("\t{}: {} - The data is {} stationary with {}% confidence"
              .format(key, value, '' if adfuller_test[0] < value else 'not', 100-int(key[:-1])))

    # Autocorrelation Plots
    fig, ax = plt.subplots(2, figsize=(14,7))
    ax[0] = plot_acf(timeseries, ax=ax[0], lags=20)
    ax[1] = plot_pacf(timeseries, ax=ax[1], lags=20)

test_stationarity(arima_df.sales)

Looking at the results from our test, we can conclude that the series is not stationary. Therefore, in order to make the series stationary we apply Differencing

Step 2: Differencing
Differencing: Seasonal or cyclical patterns can be removed by substracting periodical values. If the data is 12-month seasonal, substracting the series with a 12-lag difference series will give a “flatter” series. Since we have aggregated the data to each day-level, we will shift by 1.

In [ ]:
first_difference = arima_df.sales - arima_df.sales.shift(1)
first_difference = pd.DataFrame(first_difference.dropna(inplace=False))
# Check for stationarity after differencing
test_stationarity(first_difference.sales)

After applying Differencing to the series, we can see from the above results that the series is now stationary, i.e. mean and variance are constant over time, and from ADF we can verify that the test-statistic is lesser than the critical value, hence we can reject the null hypothesis and conclude that the series is staionary.

Step 3: Model Building
Interpreting the AR(p), I(d), MA(q) values:
Determining I(d):

Taking the first order difference makes the time series stationary. Therefore, I(d) = 1.

Determining AR(p): If the lag-1 autocorrelation of the differenced series PACF is negative, and/or there is a sharp cutoff, then choose a AR order of 1.

From the PACF plot we can clearly observe that within 6 lags the AR is significant. Therefore, we can use AR(p) = 6, (6 lines are crossed the blue lines so 6past days are required to predict).

Determining MA(q): If the lag-1 autocorrelation of the differenced series ACF is negative, and/or there is a sharp cutoff, then choose a MA order of 1.

From tha ACF plot we see a negative spike at lag 1, therfore we can use MA(q) = 1

In [ ]:
from statsmodels.tsa.arima.model import ARIMA


In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Ensure your 'sales' column is numeric and there are no missing values
arima_df['sales'] = pd.to_numeric(arima_df['sales'], errors='coerce')
arima_df = arima_df.dropna(subset=['sales'])

# Fit the ARIMA model
arima_model61 = ARIMA(arima_df['sales'], order=(6, 1, 1)).fit()
print(arima_model61.summary())


Plotting the residuals using ACF and PACF
Plotting the residuals shows that recurring correlation exists in both ACF and PACF. So we need to deal with seasonality. When the plots of ACF and PACF are similar or any sesaonality is present between them then, we need to apply the Seasonal ARIMA (SARIMA) model.

In [ ]:
residuals = arima_model61.resid
# Checking for seasonality
fig, ax = plt.subplots(2, figsize=(14,7))
ax[0] = plot_acf(residuals, ax=ax[0], lags=40)
ax[1] = plot_pacf(residuals, ax=ax[1], lags=40)

In [ ]:
# fit the model
sarima_model = SARIMAX(arima_df.sales, order=(6, 1, 0), seasonal_order=(6, 1, 0, 7),
                       enforce_invertibility=False, enforce_stationarity=False)
sarima_fit = sarima_model.fit()
arima_test_df['pred_sales'] = sarima_fit.predict(start=arima_test_df.index[0],
                                                 end=arima_test_df.index[-1], dynamic= True)
plot = sarima_fit.plot_diagnostics(figsize=(14,7))
plot

In [ ]:
arima_test_df['model'] = 'SARIMA'


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Calculate errors
arima_test_df['errors'] = arima_test_df['sales'] - arima_test_df['pred_sales']

# Update or insert the model column
arima_test_df['model'] = 'SARIMA'  # Option 1: Update existing column
# Or use Option 2:
# if 'model' in arima_test_df.columns:
#     arima_test_df.drop(columns=['model'], inplace=True)
# arima_test_df.insert(0, 'model', 'SARIMA')

# Evaluate the predictions for Seasonal ARIMA model
plt.figure(figsize=(14, 7))
plt.plot(train_df['date'], train_df['sales'], label='Train')
plt.plot(arima_test_df.index, arima_test_df['sales'], label='Test')
plt.plot(arima_test_df.index, arima_test_df['pred_sales'], label='Forecast - SARIMA')
plt.legend(loc='best')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Forecasts using Seasonal ARIMA (SARIMA) model')
plt.show()

plt.figure(figsize=(14, 7))
plt.plot(arima_test_df.index, np.abs(arima_test_df['errors']), label='Errors')
plt.plot(arima_test_df.index, arima_test_df['sales'], label='Actual Sales')
plt.plot(arima_test_df.index, arima_test_df['pred_sales'], label='Forecast')
plt.legend(loc='best')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Seasonal ARIMA (SARIMA) forecasts with actual sales and errors')
plt.show()

# Aggregate results

print(result_df_sarima)


In [ ]:
# Aggregate results without undefined functions
result_df_sarima = arima_test_df.groupby('model').agg(
    total_sales=('sales', 'sum'),
    total_pred_sales=('pred_sales', 'sum'),
    SARIMA_overall_error=('errors', 'sum'),
    MAE=('errors', lambda x: np.mean(np.abs(arima_test_df['sales'] - arima_test_df['pred_sales']))),
    RMSE=('errors', lambda x: np.sqrt(np.mean((arima_test_df['sales'] - arima_test_df['pred_sales']) ** 2))),
    MAPE=('errors', lambda x: np.mean(np.abs((arima_test_df['sales'] - arima_test_df['pred_sales']) / arima_test_df['sales'])) * 100)
)

print(result_df_sarima)

Overall, while the SARIMA model provided a reasonably close estimate of sales, it tended to overestimate the actual figures. The errors suggest areas for potential improvement, such as refining the model parameters or considering additional explanatory variables.

2.Supervised Machine Learning: Linear Regression

Let's apply Linear Regression to our time series data in order to forecasts sales.

In [ ]:
reg_df = df
reg_df

Step 1: Feature Engineering


In [ ]:
# Lag features
for i in range(1,8):
    lag_i = 'lag_' + str(i)
    reg_df[lag_i] = reg_df.sales.shift(i)

# Rolling window
reg_df['rolling_mean'] = reg_df.sales.rolling(window=7).mean()
reg_df['rolling_max'] = reg_df.sales.rolling(window=7).max()
reg_df['rolling_min'] = reg_df.sales.rolling(window=7).min()

reg_df = reg_df.dropna(how='any', inplace=False)
reg_df = reg_df.drop(['store', 'item'], axis=1)

# Split the series to predict the last 3 months of 2017
reg_df = reg_df.set_index('date')
reg_train_df = reg_df.loc[:'2017-09-30']
reg_test_df = reg_df.loc['2017-10-01':]


*Step* 2: Feature Selection and Model Building


In [ ]:
# Correlation matrix with heatmap
corr = reg_train_df.corr()
fig = plt.figure(figsize=(10,7))
_ = sn.heatmap(corr, linewidths=.5)

In [ ]:
X_train = reg_train_df.drop(['sales'], axis=1)
y_train = reg_train_df['sales'].values

X_test = reg_test_df.drop(['sales'], axis=1)
y_test = reg_test_df['sales'].values

#Univariate SelectKBest class to extract top 5 best features
top_features = SelectKBest(score_func=f_regression, k=5)
fit = top_features.fit(X_train, y_train)
df_scores = pd.DataFrame(fit.scores_)
df_columns = pd.DataFrame(X_train.columns)

#concat two dataframes for better visualization
feature_scores = pd.concat([df_columns, df_scores], axis=1)
feature_scores.columns = ['Feature','Score']  # naming the dataframe columns
print(feature_scores.nlargest(5,'Score'))  # print 5 best features

In [ ]:
# update X_train, X_test to include top features
X_train = X_train[['rolling_mean', 'rolling_max', 'rolling_min', 'lag_7', 'lag_1']]
X_test = X_test[['rolling_mean', 'rolling_max', 'rolling_min', 'lag_7', 'lag_1']]

# fit model
model = LinearRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)

errors_df = reg_test_df[['sales']]
errors_df['pred_sales'] = preds
errors_df['errors'] = preds - y_test
errors_df.insert(0, 'model', 'LinearRegression')

Step 3: Model Evaluation and Predictions


In [ ]:
import numpy as np

def mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))

def mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / actual)) * 100


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Plot actual vs predicted sales
fig = plt.figure(figsize=(14, 7))
plt.plot(reg_train_df.index, reg_train_df['sales'], label='Train')
plt.plot(reg_test_df.index, reg_test_df['sales'], label='Test')
plt.plot(errors_df.index, errors_df['pred_sales'], label='Forecast - Linear Regression')
plt.legend(loc='best')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Forecasts using Linear Regression model')
plt.show()

# Plot errors
fig = plt.figure(figsize=(14, 7))
plt.plot(errors_df.index, errors_df['errors'], label='Errors')
plt.plot(errors_df.index, errors_df['sales'], label='Actual Sales')
plt.plot(errors_df.index, errors_df['pred_sales'], label='Forecast')
plt.legend(loc='best')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Linear Regression forecasts with actual sales and errors')
plt.show()

# Aggregate results
result_df_lr = errors_df.groupby('model').agg(
    total_sales=('sales', 'sum'),
    total_pred_sales=('pred_sales', 'sum'),
    LR_overall_error=('errors', 'sum'),
    MAE=('errors', lambda x: mae(errors_df['sales'], errors_df['pred_sales'])),
    RMSE=('errors', lambda x: rmse(errors_df['sales'], errors_df['pred_sales'])),
    MAPE=('errors', lambda x: mape(errors_df['sales'], errors_df['pred_sales']))
)

print(result_df_lr)


Interpretation of Linear Regression Model Evaluation
Total Sales: The actual sales during the evaluation period amounted to 1861 units, reflecting the total sales achieved.

Total Predicted Sales: The model predicted total sales of approximately 1882.07 units. This indicates that the model estimated slightly higher sales compared to what was actually recorded.

Linear Regression Overall Error: The overall error of 21.07 suggests that the model overestimated sales by this amount, indicating a positive bias in its predictions.

Mean Absolute Error (MAE): The MAE of 3.86 indicates that, on average, the model's predictions deviated from the actual sales by about 3.86 units. This provides a measure of the typical error size.

Root Mean Square Error (RMSE): The RMSE of 4.79 indicates that the model's predictions varied from the actual sales by about 4.79 units, highlighting the spread of the errors.

Mean Absolute Percentage Error (MAPE): The MAPE of 23.11% shows that, on average, the predictions were off by approximately 23.11% relative to the actual sales figures. This percentage offers a perspective on the model's accuracy in relation to the sales volume.

Overall, the Linear Regression model provided a reasonably close estimate of sales, with a slight overestimation. The error metrics indicate that while the model's predictions are generally accurate, there is room for improvement, potentially through refining the model or incorporating additional features.





Conclusion: Model Comparison
Based on the evaluation of both the Seasonal ARIMA (SARIMA) and Linear Regression models, we can draw the following conclusions:

SARIMA Model Performance:

Total Sales: 1861
Total Predicted Sales: 1973.20
Overall Error: -112.20 (underestimation)
MAE: 4.80
RMSE: 5.84
MAPE: 29.02%
Linear Regression Model Performance:

Total Sales: 1861
Total Predicted Sales: 1882.07
Overall Error: 21.07 (overestimation)
MAE: 3.86
RMSE: 4.79
MAPE: 23.11%

Comparison Summary
Prediction Accuracy: The Linear Regression model has a lower MAE (3.86) and RMSE (4.79) compared to the SARIMA model (MAE: 4.80, RMSE: 5.84). This indicates that the Linear Regression model provides more accurate predictions on average.
Error Magnitude: The MAPE of the Linear Regression model (23.11%) is also lower than that of the SARIMA model (29.02%), suggesting it is better at forecasting sales in percentage terms.
Bias in Predictions: The SARIMA model underestimated sales, while the Linear Regression model overestimated them, indicating differing biases in the predictions.

Final Recommendation
Given the lower error metrics and better overall prediction accuracy, the Linear Regression model appears to be the better choice for forecasting sales in this case. However, it may still be beneficial to further refine both models or explore hybrid approaches to enhance forecasting performance.